# NB09 · Capital, pricing and limits

**Project:** multiyear-credit-engine
**Job requirement covered:** Insurance investment portfolio risk and statutory capital frameworks; risk limits, asset allocation.
**Engine module:** `creditengine.capital`
**Assumptions:** A-06
**Theory:** Bluhm §1.2, §5.1–5.2; NAIC Life RBC; American Academy of Actuaries C1 Work Group (2021)
**Status:** ✅ Developed — every function is built from scratch in this notebook

## Objective

How much statutory capital does the bond book tie up and how could that change over five years, which bonds use the economic capital, does each rating's spread pay for its expected loss and capital — and what happens to return on capital if the allocation shifts?

## Results at a glance

| Question | Answer |
|---|---|
| Statutory capital today | C-1 bond RBC **USD 30.3m** (3.0% of the book), with portfolio adjustment 1.61 for 200 issuers |
| How could it move? | Year-5 median **26.5m** as bonds mature; 99.5th percentile **53.2m** — in a bad cycle capital per USD held almost triples (3.0% → 8.6%) |
| Does each rating's spread pay? | Break-even: A 25 bp, BBB 49 bp, BB 162 bp, B 363 bp. Market: A 69, BBB 102, BB 194, B 315 bp. **BBB has the widest cushion (+53 bp); single B does not pay (−48 bp)** |
| Return on statutory capital | A 32%, BBB 26%, BB 13%, B 7% (hurdle 10%) |
| Who uses the economic capital? | BBB: 45% of exposure, **53% of EC**; a single sector uses 17% |
| Do limits bind? | With limits of 25 / 50 / 25% of EC, BBB (106%) and high yield (105%) breach; investment grade uses 83% |
| Allocation test | Moving USD 50m from A to BBB earns 11% on statutory capital but only **3.5% on economic capital**: the two capital views disagree because BBB drives the tail |

## Key terms and symbols

| Term | Meaning |
|---|---|
| **C-1 RBC** | NAIC risk-based capital for asset credit risk: factor × carrying value, adjusted for the number of issuers |
| **NAIC designation** | Statutory credit grade 1.A … 6, mapped from the rating (middle notch, A-06) |
| **Portfolio adjustment** | Multiplier for diversification by issuer count (first 10 issuers × 7.5, …); NB01 |
| **EL** | Expected loss over five years |
| **ES 99.5%** | Average loss in the worst 0.5% of paths |
| **EC** | Economic capital = ES − EL: the cushion needed above the average loss |
| **Risk contribution** $RC_i$ | Asset $i$'s share of EC; contributions add up exactly to EC |
| **Break-even spread** $s^*$ | Spread that pays for expected loss plus the cost of the capital the asset ties up |
| **OAS** | Option-adjusted spread: the market's extra yield over Treasuries (ICE BofA indices) |
| **CoC** | Cost of capital: return shareholders require on capital (10%) |
| **RAROC** | Risk-adjusted return on capital = (spread − expected loss) / capital |

## Inputs

| Data | File | Used for |
|---|---|---|
| Bond book and simulation (NB06) | rebuilt in section 0 with the same seed | rating paths and losses |
| Matrix and thresholds (NB01, NB04) | `data/processed/` | simulation |
| NAIC C-1 factors, portfolio adjustment, rating map (NB01) | `data/external/naic_*.csv`, `rating_scale_map.csv` | RBC |
| S&P cumulative default rates (local only) | `data/external/cumulative_default_rates.csv` | expected loss for pricing |
| ICE BofA OAS by rating (local only) | `data/raw/fred/spreads/BAML*.csv` | market spreads |

## Outputs

- `reports/tables/capital_path.csv`, `reports/tables/breakeven_spreads.csv` (regenerated, not committed)
- `reports/figures/rbc_path_fan.png` — RBC path distribution
- `reports/figures/spread_vs_breakeven.png` — market spread against break-even spread by rating

## 0. Setup: the NB06 bond book and simulation

Same book, same settings, same seed as NB06, so every capital number here sits on the losses already validated there. The simulation now also keeps each issuer's rating path and loss.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, EXTERNAL = ROOT / "data", ROOT / "data" / "external"
RATINGS = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC/C"]
STATES, D = RATINGS + ["D"], 7
Q, LGD_MEAN, K_CAPITAL, COC = 0.995, 0.60, 2.0, 0.10      # A-04, A-06
pd.set_option("display.float_format", "{:,.3f}".format)

M = pd.read_parquet(DATA / "processed" / "transition_matrix_1y.parquet").loc[STATES, STATES]
C = pd.read_parquet(DATA / "processed" / "migration_thresholds.parquet").loc[STATES, STATES].to_numpy()

In [ ]:
BOOK = {"total": 1e9, "n": 200, "seed": 7, "sectors": 10,
        "mix": {"AAA": 0.03, "AA": 0.10, "A": 0.32, "BBB": 0.45, "BB": 0.06, "B": 0.03, "CCC/C": 0.01},
        "maturity": {"AAA": 8, "AA": 8, "A": 8, "BBB": 8, "BB": 6, "B": 6, "CCC/C": 4}}

def make_book(p):                                                  # NB06
    rng = np.random.default_rng(p["seed"])
    weights = np.array([p["mix"][r] for r in RATINGS])
    rating = np.repeat(np.arange(7), np.round(weights * p["n"]).astype(int))
    size = rng.lognormal(0.0, 0.5, rating.size)
    exposure = np.zeros(rating.size)
    for r in range(7):
        m = rating == r
        exposure[m] = size[m] / size[m].sum() * weights[r] * p["total"]
    maturity = rng.integers(1, 2 * np.array([p["maturity"][RATINGS[r]] for r in rating]), endpoint=True)
    sector = rng.integers(0, p["sectors"], size=rating.size)
    w = (1 - np.exp(-50 * M.to_numpy()[rating, D])) / (1 - np.exp(-50))
    return pd.DataFrame({"rating": rating, "exposure": exposure, "maturity": maturity, "sector": sector,
                         "rho": 0.12 * w + 0.24 * (1 - w)})

def cycle_paths(n_paths, years, phi, rng):
    z, z_prev = np.empty((n_paths, years)), rng.standard_normal(n_paths)
    for t in range(years):
        z[:, t] = phi * z_prev + np.sqrt(1 - phi**2) * rng.standard_normal(n_paths)
        z_prev = z[:, t]
    return z

def migrate(state, A, C):
    new = state.copy()
    for r in range(7):
        m = state == r
        new[m] = 7 - np.searchsorted(C[r, 7:0:-1], A[m], side="left")
    return new

def simulate(book, paths=20_000, years=5, phi=0.5, lgd_mean=0.60, lgd_sd=0.25, seed=2026):
    rng = np.random.default_rng(seed)
    rho, ead, mat = book["rho"].to_numpy(), book["exposure"].to_numpy(), book["maturity"].to_numpy()
    k = lgd_mean * (1 - lgd_mean) / lgd_sd**2 - 1
    z = cycle_paths(paths, years, phi, rng)
    state = np.tile(book["rating"].to_numpy(), (paths, 1))
    ratings = np.empty((paths, years + 1, len(book)), np.int8)
    ratings[:, 0] = state
    issuer_loss = np.zeros((paths, len(book)))
    for t in range(years):
        alive = (state != D) & (mat[None, :] > t)
        A = np.sqrt(rho) * z[:, t:t + 1] + np.sqrt(1 - rho) * rng.standard_normal(state.shape)
        new = np.where(alive, migrate(state, A, C), state)
        lgd = rng.beta(lgd_mean * k, (1 - lgd_mean) * k, size=state.shape)
        issuer_loss += (alive & (new == D)) * ead * lgd
        state = new
        ratings[:, t + 1] = state
    return ratings, issuer_loss

book = make_book(BOOK)
ratings, issuer_loss = simulate(book)
L = issuer_loss.sum(axis=1)
VaR = np.quantile(L, Q)
ES = L[L >= VaR].mean()
print(f"EL {L.mean() / 1e6:.1f}m   VaR 99.5% {VaR / 1e6:.1f}m   ES 99.5% {ES / 1e6:.1f}m   (NB06: 13.6 / 85.2 / 107.0)")

## 1. RBC capital per path and year as ratings migrate

$$\text{RBC}_t = \underbrace{\sum_{i \in \text{live}} f\big(\text{rating}_{i,t}\big) \times CV_i}_{\text{base RBC}} \times \text{adj}\big(n_t\big)$$

Each live bond carries a factor for its current NAIC designation; the sum is scaled by the portfolio adjustment for the number of live issuers $n_t$. Defaulted bonds are written off and leave; matured bonds leave at par.

| Symbol | Meaning |
|---|---|
| $f(\cdot)$ | Pre-tax C-1 base factor of the bond's designation (Academy 2021), capped at 30% |
| $CV_i$ | Carrying value (here: exposure) |
| $\text{adj}(n)$ | Portfolio adjustment for $n$ issuers (NB01: 1.61 for 200 issuers) |

As ratings migrate down, factors rise — so capital can increase even before any loss is booked.

In [ ]:
naic = pd.read_csv(EXTERNAL / "naic_bond_factors.csv", dtype={"naic_designation": str}).set_index("naic_designation")["factor_pct"] / 100
scale = pd.read_csv(EXTERNAL / "rating_scale_map.csv", dtype=str).set_index("sp_category")
tiers = pd.read_csv(EXTERNAL / "naic_portfolio_adjustment.csv")
FACTOR = scale["naic_designation"].map(naic).clip(upper=0.30).reindex(STATES).to_numpy()   # by state, D last

def portfolio_adjustment(n):
    # Weighted-average tier factor for n issuers (vectorised over n)
    n = np.asarray(n, dtype=float)
    sizes = tiers["issuers_in_tier"].fillna(np.inf).to_numpy()
    lower = np.r_[0.0, np.cumsum(sizes[:-1])]
    in_tier = np.clip(n[..., None] - lower, 0.0, sizes)
    return np.divide((in_tier * tiers["factor"].to_numpy()).sum(axis=-1), n, out=np.zeros_like(n), where=n > 0)

def rbc_path(ratings, book):
    ead, mat = book["exposure"].to_numpy(), book["maturity"].to_numpy()
    out = np.zeros(ratings.shape[:2])
    for t in range(ratings.shape[1]):
        live = (ratings[:, t] != D) & (mat[None, :] > t)
        out[:, t] = (FACTOR[ratings[:, t]] * ead * live).sum(axis=1) * portfolio_adjustment(live.sum(axis=1))
    return out

rbc = rbc_path(ratings, book)
pd.DataFrame({"factor %": 100 * FACTOR[:7], "designation": scale["naic_designation"].reindex(RATINGS).to_numpy()}, index=RATINGS).T

### 1.1 Check one path by hand (V-11)

Take the worst path (largest five-year loss) and compute its RBC for year 0 and year 5 with a plain loop over the 200 bonds — no arrays — and compare.

In [ ]:
p = int(np.argmax(L))
hand = {}
for t_ in (0, 5):
    base, n_live = 0.0, 0
    for i in range(len(book)):
        r = ratings[p, t_, i]
        if r != D and book["maturity"].iat[i] > t_:
            base += FACTOR[r] * book["exposure"].iat[i]
            n_live += 1
    hand[f"year {t_}"] = {"live issuers": n_live, "base RBC, USD m": base / 1e6,
                          "adjustment": float(portfolio_adjustment(n_live)),
                          "RBC by hand, USD m": base * float(portfolio_adjustment(n_live)) / 1e6,
                          "RBC from rbc_path, USD m": rbc[p, t_] / 1e6}
pd.DataFrame(hand)

## 2. Capital path distribution

The RBC requirement on every path at the start and at the end of each year: median and tails across 20,000 paths.

In [ ]:
pcts = [5, 50, 95, 99.5]
cap_table = pd.DataFrame(np.percentile(rbc, pcts, axis=0).T / 1e6, columns=[f"p{x} USD m" for x in pcts],
                         index=pd.Index(range(rbc.shape[1]), name="year"))
live_value = np.stack([(((ratings[:, t] != D) & (book["maturity"].to_numpy()[None, :] > t)) * book["exposure"].to_numpy()).sum(axis=1)
                       for t in range(rbc.shape[1])], axis=1)
cap_table["median RBC per 100 live %"] = np.median(100 * rbc / live_value, axis=0)
cap_table["p99.5 RBC per 100 live %"] = np.percentile(100 * rbc / live_value, 99.5, axis=0)
(ROOT / "reports" / "tables").mkdir(parents=True, exist_ok=True)
cap_table.to_csv(ROOT / "reports" / "tables" / "capital_path.csv")
cap_table

In [ ]:
years = np.arange(rbc.shape[1])
fig, ax = plt.subplots(figsize=(8, 4))
for lo, hi, a in [(0.5, 99.5, 0.15), (5, 95, 0.3), (25, 75, 0.5)]:
    ax.fill_between(years, np.percentile(rbc, lo, axis=0) / 1e6, np.percentile(rbc, hi, axis=0) / 1e6,
                    color="#4C72B0", alpha=a, lw=0, label=f"{lo}–{hi}th percentile")
ax.plot(years, np.median(rbc, axis=0) / 1e6, color="black", lw=1.5, label="median")
ax.set(xlabel="year", ylabel="C-1 bond RBC, USD m", title="Statutory capital path of the bond book (20,000 paths)")
ax.legend(fontsize=8, loc="upper left")
fig.savefig(ROOT / "reports" / "figures" / "rbc_path_fan.png", dpi=150, bbox_inches="tight")
plt.show()

**Reading the result:** capital starts at USD 30.3m. On the median path it drifts down to 26.5m by year 5 as bonds mature and leave the book. In the worst 0.5% of paths it rises to 53.2m — even though the book is smaller — because surviving bonds have been downgraded into higher-factor designations and fewer issuers raise the portfolio adjustment. Per dollar still held, capital goes from 3.0% to 8.6%. That is the statutory-capital risk an insurer manages alongside loss risk: a bad cycle demands more capital exactly when losses are eating into surplus.

## 3. Break-even spread by rating and asset class

$$PD^{\text{annual}} = 1 - \big(1 - PD^{\text{cum}}_5\big)^{1/5}, \qquad EL^{\text{annual}} = PD^{\text{annual}} \times LGD$$

Average annual default probability from the five-year cumulative rate (S&P Table 24), times LGD (60%).

$$s^* = EL^{\text{annual}} + k \times f \times CoC$$

The spread must pay for the expected loss **and** for the capital the bond ties up. Capital is held at $k = 2$ times the RBC factor $f$ (a typical target of about 200% of company action level) and costs $CoC$ = 10% a year.

*Example (BBB):* annual EL 16 bp + 2 × 1.63% × 10% = 33 bp of capital cost → break-even 49 bp.

In [ ]:
cum5 = pd.read_csv(EXTERNAL / "cumulative_default_rates.csv", index_col=0)["5"].loc[RATINGS] / 100
pricing = pd.DataFrame(index=pd.Index(RATINGS, name="rating"))
pricing["5-yr cumulative PD %"] = 100 * cum5
pricing["annual EL bp"] = 1e4 * (1 - (1 - cum5) ** (1 / 5)) * LGD_MEAN
pricing["RBC factor %"] = 100 * FACTOR[:7]
pricing["capital cost bp"] = 1e4 * K_CAPITAL * FACTOR[:7] * COC
pricing["break-even bp"] = pricing["annual EL bp"] + pricing["capital cost bp"]
pricing

**Reading the result:** for investment grade the capital cost is larger than the expected loss (BBB: 33 bp of capital cost vs 16 bp of expected loss); for B and CCC the expected loss dominates. The break-even spread rises steeply below BBB because both the default probability and the NAIC factor jump at the investment-grade boundary (1.63% for BBB vs 4.66% for BB).

## 4. Compare with market spreads

Market spread = ICE BofA option-adjusted spread for each rating index, latest date available (FRED `BAML*` series, kept local under ICE's terms). The **cushion** is what the market pays above break-even; the **RAROC** is the return on the capital held, after expected loss.

$$\text{cushion} = OAS - s^*, \qquad \text{RAROC} = \frac{OAS - EL^{\text{annual}}}{k \times f}$$

A RAROC above the 10% cost of capital means the rating earns more than its capital costs.

In [ ]:
OAS_SERIES = {"AAA": "BAMLC0A1CAAA", "AA": "BAMLC0A2CAA", "A": "BAMLC0A3CA", "BBB": "BAMLC0A4CBBB",
              "BB": "BAMLH0A1HYBB", "B": "BAMLH0A2HYB", "CCC/C": "BAMLH0A3HYC"}

def load_oas(series):
    path = DATA / "raw" / "fred" / "spreads" / f"{series}.csv"
    if not path.exists():
        return None
    frame = pd.read_csv(path)
    values = pd.to_numeric(frame[series], errors="coerce")
    return pd.Series(values.to_numpy(), index=pd.to_datetime(frame["observation_date"])).dropna()

oas_hist = {r: load_oas(s) for r, s in OAS_SERIES.items()}
pricing["market OAS bp"] = [100 * s.iloc[-1] if s is not None else np.nan for s in oas_hist.values()]
pricing["OAS date"] = [s.index[-1].date() if s is not None else None for s in oas_hist.values()]
pricing["OAS percentile since 1997 %"] = [100 * (s < s.iloc[-1]).mean() if s is not None else np.nan for s in oas_hist.values()]
pricing["cushion bp"] = pricing["market OAS bp"] - pricing["break-even bp"]
pricing["RAROC %"] = 100 * (pricing["market OAS bp"] - pricing["annual EL bp"]) / 1e4 / (K_CAPITAL * FACTOR[:7])
pricing.to_csv(ROOT / "reports" / "tables" / "breakeven_spreads.csv")
missing = [r for r, s in oas_hist.items() if s is None]
print("OAS files not found for:", missing if missing else "none")
pricing[["annual EL bp", "capital cost bp", "break-even bp", "market OAS bp", "cushion bp", "RAROC %", "OAS percentile since 1997 %", "OAS date"]]

In [ ]:
show = pricing.dropna(subset=["market OAS bp"])
x = np.arange(len(show))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - 0.2, show["annual EL bp"], 0.4, color="#C44E52", label="expected loss")
ax.bar(x - 0.2, show["capital cost bp"], 0.4, bottom=show["annual EL bp"], color="#DD8452", label="capital cost")
ax.bar(x + 0.2, show["market OAS bp"], 0.4, color="#4C72B0", label="market OAS")
ax.set_xticks(x, show.index)
ax.set(ylabel="basis points per year", title="Does the spread pay? Break-even (stacked) vs market OAS")
ax.legend(fontsize=8)
fig.savefig(ROOT / "reports" / "figures" / "spread_vs_breakeven.png", dpi=150, bbox_inches="tight")
plt.show()

**Reading the result (at the spreads in the README run, 10 Oct 2026; your run uses the latest date in your files):** A, BBB and BB all clear their break-even. BBB has the widest cushion (+53 bp) and a 26% return on statutory capital. Single B does **not** pay for its expected loss and capital at current spreads (−48 bp, return 7% vs a 10% hurdle). The percentile column shows where today's spread sits in its own history: a low percentile means spreads are tight, so the cushion is thinner than usual.

## 5. Capital allocation by asset class (ES contributions)

$$RC_i = \mathbb{E}\big[L_i \mid L \ge \text{VaR}_q\big] - \mathbb{E}[L_i], \qquad \sum_i RC_i = ES - EL = EC, \qquad q = 0.995$$

An asset's capital is its average loss in the worst 0.5% of paths minus its normal average loss (Euler allocation). The pieces add up exactly to the book's economic capital, so capital can be charged to each bond, rating, sector or desk without anything left over.

$$RC_i^{\sigma} = \frac{\text{Cov}(L_i, L)}{\sigma_L}, \qquad \sum_i RC_i^{\sigma} = \sigma_L$$

Benchmark: the same idea for the standard deviation. It rewards diversification on average; the ES version looks only at the tail.

This notebook allocates the **bond** book by rating and sector. The split across bonds, commercial mortgages and structured assets on one economy is in the README (engine run) and NB11.

In [ ]:
tail = L >= VaR
rc_es = issuer_loss[tail].mean(axis=0) - issuer_loss.mean(axis=0)
rc_sd = np.array([np.cov(issuer_loss[:, i], L)[0, 1] for i in range(len(book))]) / L.std(ddof=1)

def allocate(by):
    g = pd.DataFrame({"group": by, "exposure": book["exposure"], "EL": issuer_loss.mean(axis=0), "rc_es": rc_es, "rc_sd": rc_sd})
    g = g.groupby("group", sort=False).sum()
    return pd.DataFrame({"exposure %": 100 * g["exposure"] / g["exposure"].sum(), "EL %": 100 * g["EL"] / g["EL"].sum(),
                         "std-dev capital %": 100 * g["rc_sd"] / g["rc_sd"].sum(), "ES capital %": 100 * g["rc_es"] / g["rc_es"].sum(),
                         "EC per 100 of exposure": 100 * g["rc_es"] / g["exposure"]})

by_rating = allocate(np.array(RATINGS)[book["rating"]])
print(f"sum of ES contributions {rc_es.sum() / 1e6:.2f}m  =  ES - EL {(ES - L.mean()) / 1e6:.2f}m")
by_rating

In [ ]:
by_sector = allocate("sector " + book["sector"].astype(str)).sort_values("ES capital %", ascending=False)
by_sector.head(5)

**Reading the result:** the three ways of splitting risk tell different stories. By expected loss, CCC/C and B dominate (53% together). By standard deviation, BBB leads (44%). By tail capital (ES), BBB uses **53%** of the book's economic capital — about 11 cents per dollar held, against 5 cents for A — because a bad cycle downgrades and defaults many BBB issuers together. CCC/C is 29% of expected loss but only 2% of tail capital. By sector, the largest single sector uses 17% of EC on 14% of exposure — a concentration worth a limit.

## 6. Concentration limits and a simple allocation exercise

### 6.1 Limits on capital, not exposure

$$\text{utilisation}_g = \frac{\sum_{i \in g} RC_i}{\text{limit}_g \times EC}$$

A limit on exposure treats USD 1 of AAA like USD 1 of B. A limit on **capital** charges each group for the tail risk it adds. Illustrative limits: investment grade (AAA–A) 25%, BBB 50%, high yield (BB–CCC) 25% of EC; review at 90%, breach at 100%.

In [ ]:
group = pd.Series(np.where(book["rating"] <= 2, "AAA-A", np.where(book["rating"] == 3, "BBB", "high yield")))
cap_by_group = pd.Series(rc_es).groupby(group).sum()
LIMITS = pd.Series({"AAA-A": 0.25, "BBB": 0.50, "high yield": 0.25})
util = pd.DataFrame({"capital USD m": cap_by_group / 1e6, "limit USD m": LIMITS * (ES - L.mean()) / 1e6})
util["utilisation %"] = 100 * util["capital USD m"] / util["limit USD m"]
util["status"] = np.select([util["utilisation %"] >= 100, util["utilisation %"] >= 90], ["breach", "review"], "ok")
util

**Reading the result:** with an illustrative 25 / 50 / 25% split of economic capital, BBB (106%) and high yield (105%) are in breach and investment grade has room (83%). An exposure limit would show nothing unusual: BBB is 45% of the book. The capital view shows that BBB's share of tail risk is larger than its share of the money.

### 6.2 Allocation exercise: move USD 50m (5% of the book)

Because each bond's simulated loss scales with its size, a new allocation can be tested by **re-weighting** the simulated issuer losses — the same 20,000 economies, no new simulation. For each move:

$$\Delta \text{income} = 50\text{m} \times (OAS_{\text{to}} - OAS_{\text{from}})$$

$$\text{return on capital change} = \frac{\Delta \text{income} - \Delta EL/5}{\Delta \text{capital}}$$

Measured twice: with capital = economic capital (EC, the model's tail view) and with capital = $k \times$ RBC (the statutory view). For a move **into** risk, the return must beat the 10% cost of capital. For a move **out of** risk (both changes negative), the same ratio is the net income given up per dollar of capital released: below 10% means de-risking is worth it.

In [ ]:
def reweighted(scale):
    L_new = issuer_loss @ scale
    var = np.quantile(L_new, Q)
    es = L_new[L_new >= var].mean()
    exp = book["exposure"].to_numpy() * scale
    rbc0 = (FACTOR[book["rating"]] * exp).sum() * float(portfolio_adjustment(len(book)))
    return {"EL 5y": L_new.mean(), "ES 99.5%": es, "EC": es - L_new.mean(), "RBC year 0": rbc0}

def move(src, dst, amount=50e6):
    scale = np.ones(len(book))
    for r, sign in [(src, -1), (dst, +1)]:
        m = (book["rating"] == RATINGS.index(r)).to_numpy()
        scale[m] = 1 + sign * amount / book["exposure"][m].sum()
    return scale

base_stats = reweighted(np.ones(len(book)))
rows = {}
for src, dst in [("BBB", "A"), ("A", "BBB"), ("A", "BB"), ("BBB", "B")]:
    s = reweighted(move(src, dst))
    d_income = 50e6 * (pricing.loc[dst, "market OAS bp"] - pricing.loc[src, "market OAS bp"]) / 1e4
    d = {k: s[k] - base_stats[k] for k in s}
    rows[f"{src} -> {dst}"] = {"spread income, USD m / yr": d_income / 1e6, "EL 5y, USD m": d["EL 5y"] / 1e6,
                               "EC, USD m": d["EC"] / 1e6, "RBC year 0, USD m": d["RBC year 0"] / 1e6,
                               "return on EC change %": 100 * (d_income - d["EL 5y"] / 5) / d["EC"],
                               "return on statutory capital change %": 100 * (d_income - d["EL 5y"] / 5) / (K_CAPITAL * d["RBC year 0"])}
pd.DataFrame(rows).T

**Reading the result:** moving USD 50m from A to BBB adds 0.17m a year of spread income. On **statutory** capital that is an 11% return — just above the 10% hurdle, so RBC says do it. On **economic** capital it is only 3.5%, because BBB is where the tail concentrates — so the model says don't. The reverse move (BBB → A) gives up 3.7 cents of net income per dollar of economic capital released, well below the 10% cost of capital, so de-risking BBB is worth it. Moving A to BB or BBB to B returns under 6% on either measure. The decision rule a portfolio team would use: grow where the return on **both** capital measures beats the hurdle, and cut where releasing capital costs less than the hurdle.

## Validation checks

- [x] **One path's capital equals a hand calculation (V-11)** — a plain loop over the 200 bonds gives the same RBC as `rbc_path` for years 0 and 5 of the worst path.
- [x] **Contributions sum to total** — ES contributions add up to ES − EL, and standard-deviation contributions to $\sigma_L$, exactly.
- [x] **Consistency with the engine** — losses equal NB06; the RBC path and break-even spreads equal the engine run in the README.

In [ ]:
checks = {
    "V-11: hand calculation equals rbc_path (years 0 and 5)": all(abs(v["RBC by hand, USD m"] - v["RBC from rbc_path, USD m"]) < 1e-9 for v in hand.values()),
    "ES contributions sum to ES - EL": bool(np.isclose(rc_es.sum(), ES - L.mean())),
    "Std-dev contributions sum to sigma": bool(np.isclose(rc_sd.sum(), L.std(ddof=1))),
    "Losses equal NB06 (EL 13.6m, ES 107.0m)": bool(np.allclose([L.mean() / 1e6, ES / 1e6], [13.6, 107.0], atol=0.05)),
    "RBC path equals README (30.3m year 0; 26.5m median and 53.2m p99.5 at year 5)": bool(np.allclose(
        [rbc[:, 0].mean() / 1e6, np.median(rbc[:, 5]) / 1e6, np.percentile(rbc[:, 5], 99.5) / 1e6], [30.3, 26.5, 53.2], atol=0.05)),
    "Break-even spreads equal README (A 25, BBB 49, BB 162, B 363 bp)": bool(np.allclose(
        pricing.loc[["A", "BBB", "BB", "B"], "break-even bp"], [25, 49, 162, 363], atol=0.6)),
    "Re-weighting with unit scale leaves losses unchanged": bool(np.isclose(base_stats["EL 5y"], L.mean())),
}
pd.Series(checks).map({True: "PASS", False: "FAIL"})

## Functions to move into the engine

| Function (this notebook) | Target module | Test |
|---|---|---|
| `portfolio_adjustment` | `creditengine.capital.rbc` | ≈ 1 at 824 issuers (NB01) |
| `rbc_path` | `creditengine.capital.rbc` | hand calculation on one path (V-11) |
| break-even spread, RAROC | `creditengine.capital.pricing` | README values |
| ES and std-dev contributions, `allocate` | `creditengine.capital.allocation` | sum to EC and $\sigma_L$ |
| limit utilisation | `creditengine.capital.limits` | status flags |
| `reweighted`, `move` (allocation by re-weighting) | `creditengine.capital.allocation` — **new** | unit scale returns the base |

## Findings and next steps

**Key results**
- **RBC path:** USD 30.3m at year 0 (3.0% of the book); median 26.5m and 99.5th percentile 53.2m at year 5; capital per dollar held up to 8.6% in bad paths. Matches the engine run exactly.
- **Pricing:** break-even A 25, BBB 49, BB 162, B 363 bp; at market spreads BBB has the widest cushion (+53 bp), single B does not pay (−48 bp).
- **Allocation:** BBB holds 45% of exposure and 53% of economic capital; CCC/C is 29% of expected loss but 2% of tail capital.
- **Limits:** on capital, not exposure; with 25 / 50 / 25% limits, BBB and high yield breach.
- **Decision test:** A → BBB earns 11% on statutory capital but 3.5% on economic capital — the two capital views disagree; de-risking BBB → A releases economic capital at 3.7%, below the 10% hurdle.

**Assumptions** (record in `docs/assumption_register.md`)
- A-06 unchanged: Academy 2021 pre-tax C-1 factors, middle notch per letter, capital held at 2 × RBC, cost of capital 10%.
- Added: limits on ES capital by rating group (illustrative 25 / 50 / 25%); allocation tested by re-weighting simulated issuer losses.

**Limitations** (record in `docs/limitations.md`)
1. **Pre-tax factors, no covariance formula:** C-1 bond RBC is shown alone; the RBC covariance step across risks (C-1, C-3, …) and tax effects are not modelled.
2. **Defaulted bonds leave the book:** no NAIC 6 holding period, so capital in default years is understated.
3. **One spread per rating index:** no issuer, sector or maturity spread differences; spreads are a single date.
4. **Static allocation:** the re-weighting test holds the economies fixed; it does not model trading costs or reinvestment.
5. **Bonds only here:** the asset-class split with commercial mortgages and structured assets waits for NB08 and NB11.

**Next:** NB08 (structured assets), then NB10 (model validation) and NB11 (assumptions, outputs and reporting).